# 02b_cellscape_grid_to_xenium: put a TMA core grid made on the CellScape scan onto the Xenium cells.

```text
The pathology group dearrayed the CellScape scans in QuPath and exported one table
per slide (<slide>_TMA_cores.csv: core, sector, missing, centroid_x_um / centroid_y_um,
diameter_um). Those coordinates are in the CellScape image (0.18 um pixels, file
orientation); the Xenium section is a different (serial) section on another scanner,
so the grid has to be moved onto it first. This script does that without landmarks:

  1. keep the cores of the sector(s) that are on the Xenium slide (e.g. sector A)
  2. find the orientation: all 8 combinations of 0/90/180/270 degree rotation with and
     without mirroring, plus small tilts (+-4 deg) and scale (+-6 %), each scored by how
     much Xenium tissue (cell density) falls inside the core discs; the best one wins
  3. refine with an affine fit: every core's centre is re-measured on the Xenium cells
     (mean shift), outliers dropped, repeated until stable (-> the transform in the JSON)
  4. follow the section's local warping (serial sections are not identical): every Xenium
     cell goes to its nearest core (so neighbouring cores share no cells), each core moves
     to the centre of its cells, and is kept consistent with a local affine fit to its
     ~12 neighbours (max --max-local-shift x core spacing away from it). Repeated until
     stable. Cores without tissue on Xenium sit where their neighbours put them.
  5. per core: number of Xenium cells -> missing on Xenium, next to the CellScape
     "missing" flag (both / xenium_only / cellscape_only / neither)

Outputs (default folder: <scripts>/config/core_grid/, read by 02_tma_dearray.R)
  <slide_id>.csv               grid for 02 (QuPath mode): core_name, x_um, y_um, diameter_um,
                               missing (= no tissue on Xenium) + cellscape_core, sector,
                               cellscape_missing, n_cells_xenium, shift_um, presence
  <slide_id>_cellscape_to_xenium.json   the transform (CellScape um / px -> Xenium um), also
                               usable to bring CellScape cells into the Xenium frame
  <slide_id>_grid_on_xenium.png         check plot: the grid over the Xenium cells
  <slide_id>_grid_qupath.geojson        the grid as QuPath annotations (one named circle per core,
                               class = presence) on the Xenium morphology image, for manual
                               corrections: open morphology_focus in QuPath, File > Import
                               objects (or drag the file in), move / resize / re-class cores,
                               then run qupath/export_core_edits.groovy -> <slide_id>_edited.csv.
                               02 uses <slide_id>_edited.csv instead of <slide_id>.csv when it
                               exists, so re-running 02b never overwrites manual edits.

core_name drops the sector prefix ("A_1-a" -> "1-a") so 02 matches it to tma_map.csv
by row and column like any QuPath grid; cellscape_core keeps the full name, the
`tma_core` of the CellScape per-cell tables. With several sectors (or --keep-sector-prefix)
core_name keeps it ("C_1-a", "D_1-a"), so give tma_map.csv the same core_id.

A block that is split or placed differently on the Xenium slide (or two blocks on one
slide) is fitted piece by piece: --cores (which cores) + --region (where on Xenium) +
--part (name); every run rebuilds <slide_id>.csv from all parts, e.g.
  ... --slide-id TMA_UC_CD --sector C D --cores-csv FL0035_C_TMA_cores.csv FL0035_D_TMA_cores.csv       --cores '^C_' --region 0 0 12000 11500 --part C
  ... (same) --cores '^D_' --region 0 11500 12000 17000 --part D

Example
  python 02b_cellscape_grid_to_xenium.py --slide-id TMA_ORGAN --sector A \
      --cores-csv FL0027_TMA_cores.csv --xenium-dir /path/to/output-XETG...
```

**Notebook version of `experiment1_tma/02b_cellscape_grid_to_xenium.py`.** Same code, laid out for interactive work:

1. Set the options in the **Parameters** cell (the script's command-line options: `--slide-id` becomes `slide_id`).
2. Run the cells in order. After each cell you can inspect its variables, tables and images.

Use the `.py` script for batch jobs on the cluster; both produce the same outputs. This notebook is generated by `tools/make_notebooks.py`, which overwrites it, so save your own edits under another name.

In [ ]:
import os

# Folder with config. Normally this notebook sits in
# notebooks/ next to the scripts ('..'); it is also found if the notebook is in the script
# folder itself or deeper, or if the scripts are up to 3 subfolders below the notebook.
# If not found, set SCRIPT_DIR yourself, e.g.
# SCRIPT_DIR = '/work_ikmb/<user>/Spatial_TMA/Scripts_tma'
SCRIPT_DIR = None
# a cell run while SCRIPT_DIR was still None leaves None on the import path, which breaks
# later imports (scipy: 'stat: path should be string ... not NoneType'): drop it
import sys
sys.path[:] = [p for p in sys.path if isinstance(p, str)]
_required = ['config']
_cands = [os.path.abspath('..')] + [os.path.abspath(os.path.join(*['..'] * k) if k else '.') for k in range(4)]
if SCRIPT_DIR is None:
    SCRIPT_DIR = next((d for d in _cands if all(os.path.exists(os.path.join(d, f)) for f in _required)), None)
if SCRIPT_DIR is None:
    # the scripts may also be in a subfolder (e.g. an unzipped experiment1_tma/): newest copy wins
    import glob
    _below = sorted({os.path.dirname(os.path.abspath(p)) for k in range(1, 4)
                     for p in glob.glob(os.path.join(*['*'] * k, _required[0]))
                     if '.ipynb_checkpoints' not in p},
                    key=lambda d: os.path.getmtime(os.path.join(d, _required[0])), reverse=True)
    _ok = [d for d in _below if all(os.path.exists(os.path.join(d, f)) for f in _required)]
    _cands += _below
    if _ok:
        SCRIPT_DIR = _ok[0]
        if len(_ok) > 1:
            print('Several copies of the scripts below this folder; using the newest:', *_ok, sep='\n  ')
if SCRIPT_DIR is None or not all(os.path.exists(os.path.join(SCRIPT_DIR, f)) for f in _required):
    _near = next((d for d in _cands if os.path.exists(os.path.join(d, _required[0]))), None)
    raise FileNotFoundError(
        (f'Found {_required[0]} in {_near}, but not {_required[1:]} there -- copy the py/ folder of the '
         'repository next to the scripts.' if _near else
         f'Could not find {_required[0]} in {sorted(set(_cands))}. Put this notebook in the notebooks/ '
         'folder next to the scripts, or set SCRIPT_DIR at the top of this cell.')
        + f'\n(this notebook runs in {os.getcwd()})')
print('Scripts folder:', SCRIPT_DIR)

# Which Python runs this notebook? It must be the conda environment, not the system one.
import sys
print('Python:', sys.executable)
try:
    import numpy  # noqa: F401
except ModuleNotFoundError:
    # find the environment's own python, so the fix below names the exact path
    cands = [os.path.expanduser('~/.conda/envs/exp1-imaging/bin/python'),
             os.path.join(sys.prefix, 'envs', 'exp1-imaging', 'bin', 'python')]
    cands += [os.path.join(d, 'exp1-imaging', 'bin', 'python')
              for d in os.environ.get('CONDA_ENVS_PATH', '').split(os.pathsep) if d]
    found = [c for c in cands if os.path.exists(c)]
    py = found[0] if found else '"$CONDA_PREFIX/bin/python"'
    raise ModuleNotFoundError(
        'This kernel runs ' + sys.executable + ', not the exp1-imaging environment.\n'
        'If you already picked "exp1-imaging" under Kernel > Change kernel, that kernel was registered\n'
        'with the wrong Python. Fix it once in a terminal (re-registers it with the right one):\n'
        '  conda activate exp1-imaging\n'
        f'  {py} -m pip install ipykernel\n'
        f'  {py} -m ipykernel install --user --name exp1-imaging --display-name exp1-imaging\n'
        'then reload the Jupyter page, Kernel > Change kernel > exp1-imaging, run from the top.\n'
        '(environment file: experiment1_tma/env/py_imaging.yml)')

# Packages this step needs (import name: pip name). Anything missing is installed INTO THIS
# kernel with %pip in a notebook cell -- not with pip in a terminal, which may be another Python.
import importlib.util
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'pyarrow': 'pyarrow', 'matplotlib': 'matplotlib'}
missing = [pip for mod, pip in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError(
        f'Missing in this kernel: {", ".join(missing)}\n'
        f'Run this in a new cell:  %pip install {" ".join(missing)}\n'
        'then Kernel > Restart, and run the notebook again from the top.')
print('All packages found.')

In [ ]:
import argparse
import json
import math
import os
import re
import sys

import numpy as np
import pandas as pd
from scipy import ndimage, signal
from scipy.spatial import cKDTree

HERE = SCRIPT_DIR

## Parameters

In [ ]:
from types import SimpleNamespace

# Same options as `python experiment1_tma/02b_cellscape_grid_to_xenium.py --help` (--slide-id -> slide_id).

# Options marked REQUIRED have no default: replace their None, e.g. slide_id="TMA_ORGAN".
# Paths are plain strings in quotes.

a = SimpleNamespace(
    slide_id=None,             # REQUIRED. Xenium slide_id (slides.csv), e.g. TMA_ORGAN
    cores_csv=None,            # REQUIRED. <slide>_TMA_cores.csv of the CellScape dearray; several files are combined (e.g. FL0035_C_TMA_cores.csv FL0035_D_TMA_cores.csv). a list
    xenium_dir=None,           # REQUIRED. Xenium output folder (output-XETG...) of this slide
    sector=None,               # REQUIRED. sector letter(s) on the Xenium slide: A for TMA_ORGAN (FL0027), C for TMA_TUMOR (FL0026); in the notebook as a list, e.g. ['C']. a list
    out_dir=None,              # default: <scripts>/config/core_grid
    bin_um=40.0,               # pixel size of the density image for the search
    max_tilt=4.0,              # largest extra rotation tried, degrees
    max_scale_change=0.06,     # largest scale change tried (0.06 = 6 %%)
    min_cells=30,              # cores with fewer Xenium cells inside are 'missing' on Xenium
    max_local_shift=0.35,      # a core may sit up to this x core spacing away from where its neighbours put it (0 = rigid)
    diameter_um=None,          # core diameter on Xenium; default: median CellScape diameter x scale (one size for all cores)
    cores=None,                # only cores whose CellScape name matches this regular expression, e.g. '^A_[1-7]-[a-g]$' (fit a block that was split on the Xenium slide piece by piece, with --part)
    region=None,               # only Xenium cells inside this box (um) are used for the fit, e.g. one block of the slide. a list
    part=None,                 # name of this partial fit (e.g. top, bottom, C, D): written as <slide>_part-<name>.*, and <slide>.csv / .geojson / .png are rebuilt from ALL parts of the slide
    orientation=None,          # only try this orientation (rot0, rot90, rot180, rot270, each optionally +mirror), e.g. the one found for the whole slide -- needed for parts that are square grids (7 x 7 looks the same rotated)
    keep_sector_prefix=False,  # keep 'A_' in core_name (automatic when several sectors are used, e.g. C_1-a and D_1-a)
)

missing = [k for k in ('slide_id', 'cores_csv', 'xenium_dir', 'sector') if getattr(a, k) in (None, '')]
if missing:
    raise ValueError(f'Fill in these options in the cell above (replace None with a value): {missing}')

In [ ]:
# Options not set in the Parameters cell (e.g. a cell copied from an older version
# of this notebook) get the script's default. Nothing to edit here.
_defaults = dict(
    slide_id=None,
    cores_csv=None,
    xenium_dir=None,
    sector=None,
    out_dir=None,
    bin_um=40.0,
    max_tilt=4.0,
    max_scale_change=0.06,
    min_cells=30,
    max_local_shift=0.35,
    diameter_um=None,
    cores=None,
    region=None,
    part=None,
    orientation=None,
    keep_sector_prefix=False,
)
for _k, _v in _defaults.items():
    if not hasattr(a, _k):
        setattr(a, _k, _v)
        print(f'{_k} not in the Parameters cell -> default {_v!r}')
_missing = [k for k in ('slide_id', 'cores_csv', 'xenium_dir', 'sector') if getattr(a, k, None) in (None, '')]
if _missing:
    raise ValueError(f'Still empty: {_missing}. Fill them in the Parameters cell, run THAT cell '
                     '(Shift+Enter), then this one.')
print('Options in use:', {k: v for k, v in vars(a).items() if v not in (None, False)})

## Helper functions

### Input

In [ ]:
def read_cellscape_cores(path, sectors, core_regex=None):
    paths = [path] if isinstance(path, str) else list(path)
    g = pd.concat([pd.read_csv(f) for f in paths], ignore_index=True)
    path = " + ".join(os.path.basename(f) for f in paths)
    need = {"core", "sector", "centroid_x_um", "centroid_y_um", "diameter_um"}
    if not need <= set(g.columns):
        raise ValueError(f"{path}: needs columns {sorted(need)}; found {list(g.columns)}")
    g["sector"] = g["sector"].astype(str)
    have = sorted(g["sector"].unique())
    bad = [s for s in sectors if s not in have]
    if bad:
        raise ValueError(f"sector(s) {bad} not in {os.path.basename(path)} (it has {have})")
    g = g[g["sector"].isin(sectors)].reset_index(drop=True)
    if core_regex:
        g = g[g["core"].astype(str).str.contains(core_regex, regex=True)].reset_index(drop=True)
        if g.empty:
            raise ValueError(f"no core of sector(s) {sectors} matches --cores '{core_regex}'")
    if g["core"].duplicated().any():
        raise ValueError(f"duplicated core names: {sorted(g.loc[g['core'].duplicated(), 'core'].unique())[:10]}")
    g["cellscape_missing"] = (g["missing"].astype(str).str.lower().isin(["1", "true", "t", "yes"])
                              if "missing" in g else False)
    print(f"{len(g)} cores in sector {'+'.join(sectors)} of {os.path.basename(path)} "
          f"({int(g['cellscape_missing'].sum())} flagged missing on CellScape)")
    return g

In [ ]:
def read_xenium_cells(xenium_dir):
    for name in ("cells.parquet", "cells.csv.gz"):
        f = os.path.join(xenium_dir, name)
        if os.path.isfile(f):
            cols = ["x_centroid", "y_centroid"]
            c = pd.read_parquet(f, columns=cols) if name.endswith(".parquet") else pd.read_csv(f, usecols=cols)
            xy = c.to_numpy(dtype=np.float64)
            print(f"{len(xy):,} Xenium cells from {f}")
            return xy
    raise FileNotFoundError(f"no cells.parquet / cells.csv.gz in {xenium_dir}")

### Search: orientation x tilt x scale, translation by cross-correlation

In [ ]:
def orientations():
    """The 8 axis-aligned orientations: (name, 2x2 matrix acting on column vectors)."""
    out = []
    for mirror in (False, True):
        for k in range(4):
            a = math.radians(90 * k)
            r = np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]]).round(12)
            m = r @ np.diag([-1.0, 1.0]) if mirror else r
            out.append((f"rot{90 * k}{'+mirror' if mirror else ''}", m))
    return out

In [ ]:
def rot(deg):
    a = math.radians(deg)
    return np.array([[math.cos(a), -math.sin(a)], [math.sin(a), math.cos(a)]])

In [ ]:
def density_image(xy, b, pad_um):
    """Log cell density, zero mean, padded by pad_um of 'glass' on every side (so a grid placed
    partly off the slide is not favoured over one on the tissue)."""
    x0, y0 = xy.min(0) - 2 * b
    nx = int(math.ceil((xy[:, 0].max() - x0) / b)) + 3
    ny = int(math.ceil((xy[:, 1].max() - y0) / b)) + 3
    h, _, _ = np.histogram2d(xy[:, 1], xy[:, 0], bins=[ny, nx],
                             range=[[y0, y0 + ny * b], [x0, x0 + nx * b]])   # rows = y, columns = x
    t = np.log1p(ndimage.gaussian_filter(h, 1.0))
    t /= max(np.percentile(t[t > 0], 99), 1e-9) if (t > 0).any() else 1.0
    t = np.clip(t, 0, 1)
    t -= t.mean()                                  # zero mean: discs on empty glass cost score
    pad = int(math.ceil(pad_um / b))
    t = np.pad(t, pad, constant_values=t.min())
    return t, (x0 - pad * b, y0 - pad * b)

In [ ]:
def disc_template(pts, radii, weights, b):
    t0 = pts.min(0) - radii.max() - b
    nx = int(math.ceil((pts[:, 0].max() + radii.max() - t0[0]) / b)) + 2
    ny = int(math.ceil((pts[:, 1].max() + radii.max() - t0[1]) / b)) + 2
    tpl = np.zeros((ny, nx))
    yy, xx = np.mgrid[0:ny, 0:nx]
    for (px, py), r, w in zip(pts, radii, weights):
        cx, cy, rb = (px - t0[0]) / b, (py - t0[1]) / b, r / b
        x_lo, x_hi = int(max(cx - rb - 1, 0)), int(min(cx + rb + 2, nx))
        y_lo, y_hi = int(max(cy - rb - 1, 0)), int(min(cy + rb + 2, ny))
        sub = (xx[y_lo:y_hi, x_lo:x_hi] - cx) ** 2 + (yy[y_lo:y_hi, x_lo:x_hi] - cy) ** 2 <= rb ** 2
        tpl[y_lo:y_hi, x_lo:x_hi][sub] = w
    return tpl, t0

In [ ]:
def coarse_search(P, diam, weights, img, img0, b, max_tilt, max_scale, only=None):
    c = P.mean(0)
    Pc = P - c
    tilts = np.arange(-max_tilt, max_tilt + 1e-9, 2.0) if max_tilt > 0 else np.array([0.0])
    scales = np.arange(1 - max_scale, 1 + max_scale + 1e-9, 0.02) if max_scale > 0 else np.array([1.0])
    best = None
    results = []
    cands = [(n, M) for n, M in orientations() if not only or n == only]
    if not cands:
        raise ValueError(f"--orientation {only}: use one of {[n for n, _ in orientations()]}")
    for name, M in cands:
        best_o = None
        for tilt in tilts:
            for s in scales:
                A = s * rot(tilt) @ M
                pts = Pc @ A.T
                radii = 0.45 * diam * s
                tpl, t0 = disc_template(pts, radii, weights, b)
                cc = signal.fftconvolve(img, tpl[::-1, ::-1], mode="full")
                iy, ix = np.unravel_index(int(np.argmax(cc)), cc.shape)
                score = cc[iy, ix] / tpl.sum()
                # template pixel (0,0) sits on image pixel (ix - nx + 1, iy - ny + 1)
                off = np.array([ix - tpl.shape[1] + 1, iy - tpl.shape[0] + 1])
                t = np.array(img0) + off * b - t0 - A @ c        # Xenium = A @ P + t
                cand = dict(orientation=name, tilt=float(tilt), scale=float(s), score=float(score), A=A, t=t)
                if best_o is None or score > best_o["score"]:
                    best_o = cand
        results.append(best_o)
        if best is None or best_o["score"] > best["score"]:
            best = best_o
    results.sort(key=lambda r: -r["score"])
    print("orientation search (score = tissue inside the core discs; higher is better):")
    for r in results:
        print(f"  {r['orientation']:<14} tilt {r['tilt']:+.0f} deg  scale {r['scale']:.2f}  score {r['score']:.3f}")
    if len(results) > 1 and results[1]["score"] > 0.9 * results[0]["score"]:
        print("WARNING: the two best orientations score almost the same -- check the plot carefully")
    return best, results

### Refinement on the cells

In [ ]:
def measure_centres(tree, xy, pred, radius, iters=4):
    """Mean shift of every predicted centre onto the Xenium cells around it."""
    c = pred.copy()
    n = np.zeros(len(pred), dtype=int)
    for _ in range(iters):
        for i in range(len(c)):
            idx = tree.query_ball_point(c[i], radius[i])
            n[i] = len(idx)
            if n[i]:
                c[i] = xy[idx].mean(0)
    return c, n

In [ ]:
def fit_affine(P, Q, w=None):
    X = np.column_stack([P, np.ones(len(P))])
    W = np.ones(len(P)) if w is None else w
    sol, *_ = np.linalg.lstsq(X * W[:, None], Q * W[:, None], rcond=None)
    return sol[:2].T, sol[2]                                 # A (2x2), t

In [ ]:
def refine(P, diam, xy, A, t, min_cells, spacing, rounds=5):
    tree = cKDTree(xy)
    keep = None
    for _ in range(rounds):
        pred = P @ A.T + t
        s = math.sqrt(abs(np.linalg.det(A)))
        obs, n = measure_centres(tree, xy, pred, 0.5 * diam * s)
        ok = n >= min_cells
        res = np.linalg.norm(obs - pred, axis=1)
        if ok.sum() >= 3:
            lim = max(3 * np.median(res[ok]), 0.1 * spacing)
            keep = ok & (res <= lim)
            if keep.sum() >= 3:
                A, t = fit_affine(P[keep], obs[keep])
    pred = P @ A.T + t
    s = math.sqrt(abs(np.linalg.det(A)))
    obs, n = measure_centres(tree, xy, pred, 0.5 * diam * s)
    return A, t, pred, obs, n, keep

In [ ]:
def lattice_steps(names, P):
    """The two grid steps (CellScape um) from the core names ('A_3-c' = number 3, letter c), or None."""
    m = pd.Series(names).str.extract(r"(\d+)\W*([A-Za-z]+)$")
    if m.isna().any().any():
        return None
    num = m[0].astype(int).to_numpy()
    let = m[1].str.lower().map(lambda x: sum((ord(ch) - 96) * 26 ** k for k, ch in enumerate(reversed(x)))).to_numpy()
    X = np.column_stack([np.ones(len(P)), num, let])
    if np.linalg.matrix_rank(X) < 3:
        return None
    sol = np.linalg.lstsq(X, P, rcond=None)[0]
    return sol[1], sol[2]                       # um per number step, um per letter step

In [ ]:
def grow_grid(P, xy, tree_xy, pred0, spacing_P, sp, min_cells, max_dev):
    """Place the cores outward from a well-fitting seed near the middle: each next core is
    predicted from its already placed neighbours (local affine, so the grid bends with the
    section) and moved onto its own tissue (cells closer to a placed core are not used).
    Returns positions and which cores were found on Xenium tissue."""
    n = len(P)
    near1 = cKDTree(P).query_ball_point(P, 1.5 * spacing_P)
    near3 = cKDTree(P).query_ball_point(P, 3.2 * spacing_P)
    pos = pred0.copy()
    found = np.zeros(n, dtype=bool)
    done = np.zeros(n, dtype=bool)
    r = 0.4 * sp

    def measure(c, mtree):
        k = 0
        for _ in range(6):
            idx = tree_xy.query_ball_point(c, r)
            if not idx:
                return c, 0
            pts = xy[idx]
            if mtree is not None:
                dm, _ = mtree.query(pts)
                pts = pts[dm > np.linalg.norm(pts - c, axis=1)]
            k = len(pts)
            if k == 0:
                return c, 0
            c_new = pts.mean(0)
            if np.linalg.norm(c_new - c) < 1.0:
                return c_new, k
            c = c_new
        return c, k

    def predict(i):
        q = [j for j in near3[i] if found[j] and j != i]
        if len(q) >= 3:
            Pq = P[q] - P[q].mean(0)
            sv = np.linalg.svd(Pq, compute_uv=False)
            if sv[-1] > 0.25 * sv[0]:
                w = 1.0 / (1.0 + np.linalg.norm(P[q] - P[i], axis=1) / spacing_P)
                Al, tl = fit_affine(P[q], pos[q], w)
                return Al @ P[i] + tl
        if q:
            j = min(q, key=lambda j: np.linalg.norm(P[j] - P[i]))
            return pred0[i] + (pos[j] - pred0[j])
        return pred0[i]

    seed = None
    for i in np.argsort(np.linalg.norm(P - P.mean(0), axis=1))[:30]:
        c, k = measure(pred0[i], None)
        if k >= min_cells and np.linalg.norm(c - pred0[i]) <= 0.3 * sp:
            seed = i
            pos[i], found[i], done[i] = c, True, True
            break
    if seed is None:
        return pos, found
    frontier = {j for j in near1[seed] if not done[j]}
    while frontier:
        mtree = cKDTree(pos[found])
        new = sorted(frontier, key=lambda i: -sum(found[j] for j in near1[i]))
        for i in new:
            p = predict(i)
            c, k = measure(p, mtree)
            if k >= min_cells and np.linalg.norm(c - p) <= max_dev * sp:
                pos[i], found[i] = c, True
            else:
                pos[i] = p
            done[i] = True
        frontier = {j for i in new for j in near1[i] if not done[j]}
    for i in np.where(~done)[0]:                # cores not connected to the rest
        pos[i] = predict(i)
    return pos, found

In [ ]:
def local_refine(P, xy, start, spacing, min_cells, max_dev, k_nb=12, iters=15):
    """Lloyd-style refinement constrained by the neighbours: cells -> nearest core (within
    0.6 spacing), core -> centre of its cells, unless that is more than max_dev x spacing from
    the position a local affine fit to its k_nb grid neighbours predicts."""
    n_core = len(P)
    nb = cKDTree(P).query(P, k=min(k_nb + 1, n_core))[1][:, 1:]           # neighbours in grid space
    C = start.copy()
    reach = 0.6 * spacing
    for _ in range(iters):
        d, j = cKDTree(C).query(xy, k=1, distance_upper_bound=reach)
        inside = np.isfinite(d)
        lab = j[inside]
        n = np.bincount(lab, minlength=n_core)
        sx = np.bincount(lab, weights=xy[inside, 0], minlength=n_core)
        sy = np.bincount(lab, weights=xy[inside, 1], minlength=n_core)
        good = n >= min_cells
        obs = np.column_stack([sx, sy]) / np.maximum(n, 1)[:, None]
        L = start.copy()
        for i in range(n_core):
            q = nb[i][good[nb[i]]]
            if len(q) >= 4:
                w = 1.0 / (1.0 + np.linalg.norm(P[q] - P[i], axis=1) / spacing)
                Al, tl = fit_affine(P[q], obs[q], w)
                L[i] = Al @ P[i] + tl
        ok = good & (np.linalg.norm(obs - L, axis=1) <= max_dev * spacing)
        newC = np.where(ok[:, None], obs, L)
        moved = float(np.abs(newC - C).max())
        C = newC
        if moved < 1.0:
            break
    return C, ok, L

In [ ]:
def xenium_pixel_size(xenium_dir):
    f = os.path.join(xenium_dir, "experiment.xenium")
    try:
        with open(f) as fh:
            return float(json.load(fh)["pixel_size"])
    except Exception:
        return 0.2125

In [ ]:
PRESENCE_COLOURS = {"both": [242, 194, 0], "xenium_only": [47, 168, 79], "cellscape_only": [208, 48, 48],
                    "neither": [154, 154, 154]}

In [ ]:
def write_qupath_geojson(out, px_um, path, n_vertices=48):
    """Cores as QuPath annotations (circles) in pixels of the Xenium morphology image."""
    th = np.linspace(0, 2 * np.pi, n_vertices, endpoint=False)
    feats = []
    for r in out.itertuples(index=False):
        rad = r.diameter_um / 2 / px_um
        cx, cy = r.x_um / px_um, r.y_um / px_um
        ring = [[float(cx + rad * np.cos(t)), float(cy + rad * np.sin(t))] for t in th]
        ring.append(ring[0])
        feats.append({"type": "Feature",
                      "geometry": {"type": "Polygon", "coordinates": [ring]},
                      "properties": {"objectType": "annotation", "name": str(r.core_name),
                                     "classification": {"name": r.presence,
                                                        "color": PRESENCE_COLOURS.get(r.presence, [200, 200, 200])},
                                     "isLocked": False}})
    with open(path, "w") as fh:
        json.dump({"type": "FeatureCollection", "features": feats}, fh)

In [ ]:
def describe(A):
    """Rotation (deg), mirrored, scale x / y of a 2x2 matrix."""
    det = np.linalg.det(A)
    mirrored = det < 0
    B = A @ np.diag([-1.0, 1.0]) if mirrored else A
    angle = math.degrees(math.atan2(B[1, 0], B[0, 0]))
    return dict(rotation_deg=round(angle, 2), mirrored=bool(mirrored),
                scale_x=round(float(np.linalg.norm(A[:, 0])), 4), scale_y=round(float(np.linalg.norm(A[:, 1])), 4))

### Plot

In [ ]:
def plot(xy, out, slide_id, sectors, info):
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle

    colours = {"both": "#f2c200", "xenium_only": "#2fa84f", "cellscape_only": "#d03030", "neither": "#9a9a9a"}
    span = xy.max(0) - xy.min(0)
    w = 14
    fig, ax = plt.subplots(figsize=(w, max(4, w * span[1] / max(span[0], 1))))
    b = 20.0
    h, xe, ye = np.histogram2d(xy[:, 0], xy[:, 1], bins=[max(1, int(span[0] / b)), max(1, int(span[1] / b))])
    ax.imshow(np.log1p(h.T), extent=[xe[0], xe[-1], ye[-1], ye[0]], cmap="gray", interpolation="nearest")
    for r in out.itertuples():
        col = colours[r.presence]
        ax.add_patch(Circle((r.x_um, r.y_um), r.diameter_um / 2, fill=False, lw=0.8, ec=col))
        ax.text(r.x_um, r.y_um, r.core_name, color=col, fontsize=5, ha="center", va="center")
    ax.set_xlim(min(xe[0], out.x_um.min() - 600), max(xe[-1], out.x_um.max() + 600))
    ax.set_ylim(max(ye[-1], out.y_um.max() + 600), min(ye[0], out.y_um.min() - 600))     # y down, as Xenium
    ax.set_aspect("equal")
    ax.set_xlabel("x (um, Xenium)")
    ax.set_ylabel("y (um, Xenium)")
    counts = out["presence"].value_counts()
    ax.set_title(f"{slide_id}: CellScape sector {'+'.join(sectors)} on the Xenium cells  "
                 f"[{info['orientation']}, rotation {info['rotation_deg']:.1f} deg, scale {info['scale_x']:.3f}]\n"
                 + "   ".join(f"{k}: {counts.get(k, 0)}" for k in colours)
                 + "   (yellow = tissue on both, green = Xenium only, red = CellScape only, grey = neither)",
                 fontsize=9)
    fig.tight_layout()
    return fig

## Run

In [ ]:
out_dir = a.out_dir or os.path.join(HERE, "config", "core_grid")
os.makedirs(out_dir, exist_ok=True)

if isinstance(a.sector, str):          # notebook: sector='C' or 'A, B' instead of a list
    a.sector = [t for t in re.split(r"[\s,;]+", a.sector.strip()) if t]
if isinstance(a.cores_csv, str):       # notebook: one path as a string
    a.cores_csv = [a.cores_csv]
for k, v in (("cores", None), ("region", None), ("part", None), ("keep_sector_prefix", False),
             ("orientation", None)):
    if not hasattr(a, k):              # older notebook parameter cells
        setattr(a, k, v)
g = read_cellscape_cores(a.cores_csv, a.sector, a.cores)
xy = read_xenium_cells(a.xenium_dir)
if a.region:
    x0, y0, x1, y1 = a.region
    inside = (xy[:, 0] >= x0) & (xy[:, 0] <= x1) & (xy[:, 1] >= y0) & (xy[:, 1] <= y1)
    print(f"--region {a.region}: {int(inside.sum()):,} of {len(xy):,} Xenium cells used for the fit")
    if inside.sum() < 1000:
        raise ValueError("fewer than 1000 Xenium cells inside --region -- check the coordinates (um)")
    xy = xy[inside]
P = g[["centroid_x_um", "centroid_y_um"]].to_numpy(dtype=np.float64)
cs_diam = g["diameter_um"].to_numpy(dtype=np.float64)
# one size for all cores: the CellScape ellipses vary with how each one was drawn
diam = np.full(len(P), float(np.median(cs_diam)))
spacing = float(np.median(cKDTree(P).query(P, k=2)[0][:, 1]))
weights = np.where(g["cellscape_missing"], 0.5, 1.0)

In [ ]:
img, img0 = density_image(xy, a.bin_um, pad_um=float(np.ptp(P, axis=0).max()) * 0.6)
best, _ = coarse_search(P, diam, weights, img, img0, a.bin_um, a.max_tilt, a.max_scale_change, a.orientation)
A, t, pred, obs, n, keep = refine(P, diam, xy, best["A"], best["t"], a.min_cells, spacing)

In [ ]:
s = math.sqrt(abs(np.linalg.det(A)))
d_um = a.diameter_um or float(diam[0] * s)
tree = cKDTree(xy)
sp = spacing * s

In [ ]:
def coverage(C):
    return float(np.mean(cKDTree(C).query(xy, k=1)[0] <= 0.5 * d_um))

if a.max_local_shift > 0:
    # the grid is periodic: also try it one row / column off and keep the placement whose
    # 10 x 20 (or whatever) grid covers most of the Xenium tissue -- the TMA edges decide
    steps = lattice_steps(g["core"].astype(str), P)
    shifts = [(0, 0)] + ([(i, j) for i in (-1, 0, 1) for j in (-1, 0, 1) if (i, j) != (0, 0)] if steps else [])
    cands = []
    for i, j in shifts:
        delta = (i * steps[0] + j * steps[1]) if steps else np.zeros(2)
        pos, found = grow_grid(P, xy, tree, pred + A @ delta, spacing, sp, a.min_cells, a.max_local_shift)
        cands.append(dict(shift=(i, j), pos=pos, found=found, cov=coverage(pos)))
    cands.sort(key=lambda c: -c["cov"])
    print("grid position (number step, letter step) -> Xenium cells covered by the cores:")
    for c in cands:
        print(f"  shift {str(c['shift']):<9} {100 * c['cov']:.1f}%  ({int(c['found'].sum())} cores on tissue)")
    if len(cands) > 1 and cands[1]["cov"] > cands[0]["cov"] - 0.01:
        print("WARNING: two grid positions cover the tissue almost equally -- check the core labels on the plot")
    best_c = cands[0]
    final, local, _ = local_refine(P, xy, best_c["pos"], sp, a.min_cells, 0.25)
    A, t = fit_affine(P[local], final[local]) if local.sum() >= 3 else (A, t)
    pred = P @ A.T + t
else:
    final, local = pred.copy(), np.zeros(len(P), dtype=bool)
dist = np.linalg.norm(final - pred, axis=1)
n_final = np.array([len(tree.query_ball_point(c, 0.5 * d_um)) for c in final])
xen_missing = n_final < a.min_cells
cs_missing = g["cellscape_missing"].to_numpy()
presence = np.select([~xen_missing & ~cs_missing, ~xen_missing & cs_missing, xen_missing & ~cs_missing],
                     ["both", "xenium_only", "cellscape_only"], "neither")

In [ ]:
out = pd.DataFrame({
    "core_name": (g["core"].astype(str) if (a.keep_sector_prefix or len(a.sector) > 1)
                  else g["core"].astype(str).str.replace(r"^[A-Za-z]+_", "", regex=True)),
    "x_um": final[:, 0].round(2), "y_um": final[:, 1].round(2),
    "diameter_um": round(d_um, 1),
    "missing": xen_missing,
    "cellscape_core": g["core"].astype(str), "sector": g["sector"],
    "cellscape_missing": cs_missing,
    "n_cells_xenium": n_final,
    "shift_um": dist.round(1),
    "cellscape_diameter_um": cs_diam.round(1),
    "presence": presence,
})

In [ ]:
# how much of the Xenium tissue the grid covers (cells within 0.6 x diameter of a core centre)
near = cKDTree(final).query(xy, k=1)
covered = float(np.mean(near[0] <= 0.6 * d_um))
fit_res = np.linalg.norm(obs[keep] - pred[keep], axis=1) if keep is not None and keep.any() else np.array([np.nan])
info = dict(orientation=best["orientation"], **describe(A))
info.update(
    slide_id=a.slide_id, sectors=a.sector, cellscape_cores_csv=[os.path.abspath(f) for f in a.cores_csv],
    cores_regex=a.cores, region_um=a.region, part=a.part,
    xenium_dir=os.path.abspath(a.xenium_dir),
    note="x_xenium_um = A @ [x_cellscape_um, y_cellscape_um] + t ; CellScape um = CellScape file pixels x 0.18 "
         "(file orientation, not the mirrored QuPath view)",
    A=A.round(8).tolist(), t=t.round(4).tolist(),
    cores_used_for_fit=int(keep.sum()) if keep is not None else 0,
    fit_residual_um_median=round(float(np.median(fit_res)), 1),
    fit_residual_um_p90=round(float(np.percentile(fit_res, 90)), 1),
    core_spacing_um=round(spacing * s, 1),
    xenium_cells_inside_grid=round(covered, 3),
    search_score=round(best["score"], 4),
)

In [ ]:
stem = a.slide_id + (f"_part-{a.part}" if a.part else "")
grid_csv = os.path.join(out_dir, f"{stem}.csv")
out.to_csv(grid_csv, index=False)
with open(os.path.join(out_dir, f"{stem}_cellscape_to_xenium.json"), "w") as fh:
    json.dump(info, fh, indent=2)
px_um = xenium_pixel_size(a.xenium_dir)
fig = plot(xy, out, a.slide_id + (f" (part {a.part})" if a.part else ""), a.sector, info)
png = os.path.join(out_dir, f"{stem}_grid_on_xenium.png")
fig.savefig(png, dpi=200)
if a.part:
    # the slide's grid = all its parts (each core from the part that fitted it last)
    import glob
    parts = sorted(glob.glob(os.path.join(out_dir, f"{a.slide_id}_part-*.csv")), key=os.path.getmtime)
    allg = pd.concat([pd.read_csv(f) for f in parts], ignore_index=True)
    allg = allg.drop_duplicates("cellscape_core", keep="last").reset_index(drop=True)
    dup = allg["core_name"][allg["core_name"].duplicated()]
    if len(dup):
        print(f"WARNING: the same core_name in several parts: {sorted(dup.unique())[:10]} -- "
              "use --keep-sector-prefix")
    allg.to_csv(os.path.join(out_dir, f"{a.slide_id}.csv"), index=False)
    print(f"{a.slide_id}.csv rebuilt from {len(parts)} part(s): "
          + ", ".join(os.path.basename(f) for f in parts) + f" -> {len(allg)} cores")
    xy_all = read_xenium_cells(a.xenium_dir)
    info_all = dict(info, orientation="+".join(sorted({json.load(open(f.replace(".csv", "_cellscape_to_xenium.json")))
                                                        ["orientation"] for f in parts})))
    figa = plot(xy_all, allg, a.slide_id + " (all parts)", sorted(allg["sector"].astype(str).unique()), info_all)
    figa.savefig(os.path.join(out_dir, f"{a.slide_id}_grid_on_xenium.png"), dpi=200)
    write_qupath_geojson(allg, px_um, os.path.join(out_dir, f"{a.slide_id}_grid_qupath.geojson"))
else:
    write_qupath_geojson(out, px_um, os.path.join(out_dir, f"{a.slide_id}_grid_qupath.geojson"))
gj = os.path.join(out_dir, f"{a.slide_id}_grid_qupath.geojson")

In [ ]:
counts = out["presence"].value_counts()
print(f"\nCellScape -> Xenium: {info['orientation']} (rotation {info['rotation_deg']} deg, "
      f"mirrored {info['mirrored']}), scale {info['scale_x']:.3f} / {info['scale_y']:.3f}")
print(f"affine fit on {info['cores_used_for_fit']} cores: residual median {info['fit_residual_um_median']} um, "
      f"90% {info['fit_residual_um_p90']} um (core spacing {info['core_spacing_um']} um)")
print(f"local refinement: {int(local.sum())} cores placed on their own Xenium tissue; distance from the affine "
      f"grid median {np.median(dist):.0f} um, max {dist.max():.0f} um; core diameter {d_um:.0f} um")
print("cores: " + ", ".join(f"{k} {counts.get(k, 0)}" for k in ("both", "xenium_only", "cellscape_only", "neither")))
print(f"{100 * covered:.0f}% of the Xenium cells lie on a grid core")
if covered < 0.7:
    print("WARNING: much Xenium tissue is outside the grid -- wrong sector, or the fit is off. Check the plot.")
if info["fit_residual_um_median"] > 0.15 * info["core_spacing_um"]:
    print("WARNING: large fit residuals -- check the plot before using the grid.")
edited = os.path.join(out_dir, f"{a.slide_id}_edited.csv")
if os.path.isfile(edited):
    print(f"NOTE: {os.path.basename(edited)} (your manual corrections) exists -- 02 uses it, not this new grid. "
          "Delete it to use the new grid.")
print(f"\nwritten: {grid_csv}\n         {png}\n         {gj}  (QuPath, for manual corrections)\n"
      f"Next: 02_tma_dearray.R (QuPath grid mode picks up {a.slide_id}.csv"
      f"{', built from all parts -- fit every part first' if a.part else ''}); its core names "
      f"('{out['core_name'].iloc[0]}') are matched to tma_map.csv by row + column.")